# 04 — Sentiment Aggregation & JST Join

**Purpose:**
1. Map BIS speeches to the 18 JST country ISO codes
2. Aggregate per-speech scores → annual country-year averages
3. Build lagged sentiment features (t−1)
4. Join to JST macro panel → master modelling dataset

| Path | File |
|---|---|
| Input 1 | `data/processed/bis_sentiment_raw.csv` |
| Input 2 | `data/raw/JSTdatasetR6.xlsx` |
| Output 1 | `data/processed/sentiment_annual.csv` |
| Output 2 | `data/processed/jst_sentiment_master.csv` |

---

## Cell 1 — Paths and imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy  as np

# ── YOUR EXACT PATHS ──────────────────────────────────────────────
BASE_DIR = Path(r'C:\Users\Owner\OneDrive\dissertation')
# ─────────────────────────────────────────────────────────────────

RAW_DIR        = BASE_DIR / 'data' / 'raw'
PROC_DIR       = BASE_DIR / 'data' / 'processed'
BIS_DIR        = RAW_DIR  / 'Bis_Org_Speaches'

RAW_SENTIMENT  = PROC_DIR / 'bis_sentiment_raw.csv'
JST_FILE       = RAW_DIR  / 'JSTdatasetR6.xlsx'
OUT_ANNUAL     = PROC_DIR / 'sentiment_annual.csv'
OUT_MASTER     = PROC_DIR / 'jst_sentiment_master.csv'

PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Input file check:')
for p in [RAW_SENTIMENT, JST_FILE]:
    tag = '✅  found' if p.exists() else '❌  MISSING'
    print(f'  {tag}  →  {p.name}')
print()
print('Output files will be written to:')
print(f'  {OUT_ANNUAL}')
print(f'  {OUT_MASTER}')

Input file check:
  ✅  found  →  bis_sentiment_raw.csv
  ✅  found  →  JSTdatasetR6.xlsx

Output files will be written to:
  C:\Users\Owner\OneDrive\dissertation\data\processed\sentiment_annual.csv
  C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv


C:\Users\Owner\AppData\Local\Temp\ipykernel_13708\855075939.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Cell 2 — Load raw sentiment output from Notebook 03

In [5]:
df_raw = pd.read_csv(RAW_SENTIMENT)

print(f'Raw sentiment rows  : {len(df_raw):,}')
print(f'Year range          : {df_raw["year"].min()} - {df_raw["year"].max()}')
print(f'Missing values      : {df_raw[["P_pos","P_neg","P_neutral"]].isna().sum().to_dict()}')
print()
print('Columns available:')
print(df_raw.columns.tolist())
print()
print('Sample data (first 5 rows):')
print(df_raw.head().to_string(index=False))
# print('Sample filenames (first 5):')
# print(df_raw['filename'].head().to_string(index=False))

Raw sentiment rows  : 16,622
Year range          : 1997 - 2020
Missing values      : {'P_pos': 4077, 'P_neg': 4077, 'P_neutral': 4077}

Columns available:
['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral']

Sample data (first 5 rows):
                                    url  year                date             author                                                                                                                                                                                         description    P_pos    P_neg  P_neutral
https://www.bis.org/review/r970512a.pdf  1997 1997-04-24 00:00:00   Laurence H Meyer                                              Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97. 0.132078 0.230044   0.637878
https://www.bis.org/review/r970605b.pdf  1997 1997-05-26 00:00:00    Lars Heikensten                                

In [11]:
print("Columns in df_raw:")
print(df_raw.columns.tolist())
print()
print("Sample rows:")
print(df_raw.head(3).to_string())
print()
print("First column sample:")
print(df_raw.iloc[:, 0].head(10).tolist())

Columns in df_raw:
['url', 'year', 'date', 'author', 'description', 'P_pos', 'P_neg', 'P_neutral']

Sample rows:
                                       url  year                 date              author                                                                                                                                                                                          description     P_pos     P_neg  P_neutral
0  https://www.bis.org/review/r970512a.pdf  1997  1997-04-24 00:00:00    Laurence H Meyer                                               Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US Federal Reserve System, at the Forecasters Club of New York on 24/4/97.  0.132078  0.230044   0.637878
1  https://www.bis.org/review/r970605b.pdf  1997  1997-05-26 00:00:00     Lars Heikensten                                                                Address by the Deputy Governor of the Bank of Sweden, Mr. Lars Heikensten, at the Monetary Policy 

In [13]:
## # Extract ISO country codes from description column
bank_to_iso = {
    'Federal Reserve': 'US', 'Bank of England': 'GB',
    'Bank of Japan': 'JP', 'Bank of Sweden': 'SE',
    'Bank of Australia': 'AU', 'Reserve Bank of Australia': 'AU',
    'Bank of Canada': 'CA', 'Bank of France': 'FR',
    'Deutsche Bundesbank': 'DE', 'Bank of Italy': 'IT',
    'Bank of Spain': 'ES', 'Netherlands Bank': 'NL',
    'Swiss National Bank': 'CH', 'Bank of Norway': 'NO',
    'Bank of Finland': 'FI', 'Bank of Belgium': 'BE',
    'Reserve Bank of India': 'IN', 'Bank of Korea': 'KR',
    'Reserve Bank of New Zealand': 'NZ', 'Bank of Mexico': 'MX',
    'European Central Bank': 'EU', 'Bank for International': 'BIS'
}

def extract_iso(description):
    if pd.isna(description):
        return None
    for bank, iso in bank_to_iso.items():
        if bank.lower() in str(description).lower():
            return iso
    return None

df_raw['iso'] = df_raw['description'].apply(extract_iso)

print("ISO extraction results:")
print(df_raw['iso'].value_counts())
print(f"\nUnmapped speeches: {df_raw['iso'].isna().sum()}")

ISO extraction results:
iso
EU     2123
US     2089
IN      786
DE      725
JP      674
GB      655
CA      504
AU      482
CH      370
FR      324
IT      304
ES      283
NO      241
NZ      169
FI      158
NL      146
KR       86
MX       78
BIS      73
BE       43
SE       31
Name: count, dtype: int64

Unmapped speeches: 6278


In [15]:
## # Extended bank to ISO mapping for remaining speeches
bank_to_iso_extended = {
    'Bank of Greece': 'GR', 'Bank of Portugal': 'PT',
    'Bank of Austria': 'AT', 'Bank of Denmark': 'DK',
    'Bank of Ireland': 'IE', 'Bank of Thailand': 'TH',
    'Bank of Indonesia': 'ID', 'Bank of Malaysia': 'MY',
    'Bank of China': 'CN', "People's Bank of China": 'CN',
    'Bank of Brazil': 'BR', 'Central Bank of Brazil': 'BR',
    'Bank of Turkey': 'TR', 'Central Bank of Turkey': 'TR',
    'Bank of Russia': 'RU', 'Central Bank of Russia': 'RU',
    'Bank of Poland': 'PL', 'National Bank of Poland': 'PL',
    'Bank of Hungary': 'HU', 'Magyar Nemzeti Bank': 'HU',
    'Czech National Bank': 'CZ', 'Bank of Israel': 'IL',
    'South African Reserve Bank': 'ZA', 'Bank of Argentina': 'AR',
    'Central Bank of Chile': 'CL', 'Bank of Chile': 'CL',
    'Central Bank of Philippines': 'PH', 'Bangko Sentral': 'PH',
    'Reserve Bank of South Africa': 'ZA',
    'Monetary Authority of Singapore': 'SG',
    'Hong Kong Monetary Authority': 'HK',
}

def extract_iso_extended(description):
    if pd.isna(description):
        return None
    desc_lower = str(description).lower()
    # First try original mapping
    for bank, iso in bank_to_iso.items():
        if bank.lower() in desc_lower:
            return iso
    # Then try extended mapping
    for bank, iso in bank_to_iso_extended.items():
        if bank.lower() in desc_lower:
            return iso
    return None

df_raw['iso'] = df_raw['description'].apply(extract_iso_extended)

print("Updated ISO extraction results:")
print(df_raw['iso'].value_counts())
print(f"\nUnmapped speeches: {df_raw['iso'].isna().sum()}")
print(f"Mapped speeches: {df_raw['iso'].notna().sum()}")

Updated ISO extraction results:
iso
EU     2123
US     2089
IN      786
DE      725
JP      674
GB      655
CA      504
AU      482
MY      457
PH      415
ZA      373
CH      370
FR      324
IT      304
ES      283
SG      242
NO      241
HK      218
TH      217
IE      213
NZ      169
FI      158
NL      146
GR      143
CN      120
CL      106
IL      103
DK       94
KR       86
MX       78
BIS      73
PT       55
CZ       50
BE       43
SE       31
AR       31
RU       27
PL       26
BR       12
HU       12
TR        4
ID        2
Name: count, dtype: int64

Unmapped speeches: 3358
Mapped speeches: 13264


## Cell 3 — Extract country ISO from BIS filename

BIS speech filenames follow the pattern **`r` + `YY` + `MMDD` + `letter`**, e.g.:
- `r970103a.txt` → 1997-01-03, speech 'a'
- `r080912e.txt` → 2008-09-12, speech 'e'

The **country / institution** is identified from a companion metadata CSV that BIS provides alongside the bulk download. This cell loads it if present, or guides you to download it.

In [19]:
# Full institution → ISO lookup for all 18 JST countries
INSTITUTION_TO_ISO = {
    # USA
    'federal reserve':              'USA',
    'board of governors':           'USA',
    'federal open market':          'USA',
    # UK
    'bank of england':              'GBR',
    # Germany
    'deutsche bundesbank':          'DEU',
    'bundesbank':                   'DEU',
    # France
    'banque de france':             'FRA',
    # Italy
    'banca d italia':               'ITA',
    "banca d'italia":               'ITA',
    # Spain
    'banco de espana':              'ESP',
    'banco de españa':              'ESP',
    # Netherlands
    'nederlandsche bank':           'NLD',
    # Belgium
    'national bank of belgium':     'BEL',
    'banque nationale de belgique': 'BEL',
    # Portugal
    'banco de portugal':            'PRT',
    # Ireland
    'central bank of ireland':      'IRL',
    # Switzerland
    'swiss national bank':          'CHE',
    # Japan
    'bank of japan':                'JPN',
    # Australia
    'reserve bank of australia':    'AUS',
    # Canada
    'bank of canada':               'CAN',
    # Sweden
    'riksbank':                     'SWE',
    'sveriges riksbank':            'SWE',
    # Norway
    'norges bank':                  'NOR',
    # Denmark
    'danmarks nationalbank':        'DNK',
    # Finland
    'bank of finland':              'FIN',
    'suomen pankki':                'FIN',
    # ECB — proxy-assigned to DEU (adjust if preferred)
    'european central bank':        'DEU',
}

JST_ISOS = ['USA','GBR','DEU','FRA','ITA','ESP','NLD','BEL',
            'PRT','IRL','CHE','JPN','AUS','CAN','SWE','NOR','DNK','FIN']


def match_iso(name):
    if pd.isna(name):
        return None
    name_l = str(name).lower()
    for key, iso in INSTITUTION_TO_ISO.items():
        if key in name_l:
            return iso
    return None


# Try to find BIS metadata CSV in the Bis_Org_Speaches folder
possible_meta = list(BIS_DIR.glob('*.csv'))
META_FILE = possible_meta[0] if possible_meta else None

if META_FILE:
    print(f'BIS metadata found: {META_FILE.name}')
    meta = pd.read_csv(META_FILE, encoding='utf-8', errors='ignore')
    meta.columns = meta.columns.str.lower().str.strip()
    print(f'Columns: {list(meta.columns)}')
    has_meta = True
else:
    print('⚠️  No BIS metadata CSV found in Bis_Org_Speaches\\')
    print()
    print('To get it:')
    print('  1. Go to: https://www.bis.org/cbspeeches/download.htm')
    print('  2. Download the spreadsheet / CSV of speech metadata')
    print(f'  3. Save it to: {BIS_DIR}')
    print()
    print('The metadata CSV contains columns: filename, date, author, centralbank, title')
    print('Continuing without it — iso column will be NULL until metadata is added.')
    has_meta = False

⚠️  No BIS metadata CSV found in Bis_Org_Speaches\

To get it:
  1. Go to: https://www.bis.org/cbspeeches/download.htm
  2. Download the spreadsheet / CSV of speech metadata
  3. Save it to: C:\Users\Owner\OneDrive\dissertation\data\raw\Bis_Org_Speaches

The metadata CSV contains columns: filename, date, author, centralbank, title
Continuing without it — iso column will be NULL until metadata is added.


In [21]:
## # We extracted iso codes from description column — override has_meta
has_meta = True
print(f"has_meta overridden to: {has_meta}")
print(f"ISO codes available: {df_raw['iso'].notna().sum()} speeches mapped")
print(f"Sample iso values: {df_raw['iso'].value_counts().head(5).to_dict()}")

has_meta overridden to: True
ISO codes available: 13264 speeches mapped
Sample iso values: {'EU': 2123, 'US': 2089, 'IN': 786, 'DE': 725, 'JP': 674}


In [25]:
## # Create meta dataframe from df_raw since we have no BIS metadata file
import pandas as pd

meta = pd.DataFrame({
    'filename': df_raw['url'].str.extract(r'/(r\w+\.pdf)$')[0],
    'date': df_raw['date'],
    'author': df_raw['author'],
    'centralbank': df_raw['description'].str[:80],
    'iso': df_raw['iso'],
    'year': df_raw['year']
})

print("Meta dataframe created:")
print(f"Rows: {len(meta)}")
print(f"Columns: {meta.columns.tolist()}")
print()
print(meta.head(3).to_string())

Meta dataframe created:
Rows: 16622
Columns: ['filename', 'date', 'author', 'centralbank', 'iso', 'year']

       filename                 date              author                                                                       centralbank iso  year
0  r970512a.pdf  1997-04-24 00:00:00    Laurence H Meyer  Remarks by Mr. Laurence H. Meyer, a member of the Board of Governors of the US F  US  1997
1  r970605b.pdf  1997-05-26 00:00:00     Lars Heikensten  Address by the Deputy Governor of the Bank of Sweden, Mr. Lars Heikensten, at th  SE  1997
2  r971104a.pdf  1997-10-28 00:00:00  Graeme J. Thompson  Speech by the Deputy Governor of the Reserve Bank of Australia, Mr. G.J. Thompso  AU  1997


## Cell 4 — Attach ISO codes to raw sentiment scores

In [31]:
# Cell 4 — Attach ISO codes to raw sentiment scores (fully self-contained)

# Create df_merged from df_raw
df_merged = df_raw.copy()

# Extract filename from url if not already present
if 'filename' not in df_merged.columns:
    df_merged['filename'] = df_merged['url'].str.extract(r'/(r\w+\.pdf)$')[0]

# Clean filename
df_merged['filename'] = df_merged['filename'].fillna('').str.strip()

# Ensure iso column exists
if 'iso' not in df_merged.columns:
    df_merged['iso'] = None

# Report on ISO matching
matched = df_merged['iso'].notna().sum()
total   = len(df_merged)
pct     = 100 * matched / total if total > 0 else 0
print(f'ISO matched: {matched:,} / {total:,} speeches ({pct:.1f}%)')

# Show breakdown by ISO
print()
iso_counts = df_merged.groupby('iso').size().sort_values(ascending=False)
print(iso_counts.to_string())

ISO matched: 13,264 / 16,622 speeches (79.8%)

iso
EU     2123
US     2089
IN      786
DE      725
JP      674
GB      655
CA      504
AU      482
MY      457
PH      415
ZA      373
CH      370
FR      324
IT      304
ES      283
SG      242
NO      241
HK      218
TH      217
IE      213
NZ      169
FI      158
NL      146
GR      143
CN      120
CL      106
IL      103
DK       94
KR       86
MX       78
BIS      73
PT       55
CZ       50
BE       43
SE       31
AR       31
RU       27
PL       26
BR       12
HU       12
TR        4
ID        2


In [35]:
## # Diagnostic — check JST country codes vs our ISO codes
import pandas as pd

# Load JST to see its country codes
jst = pd.read_excel('C:\\Users\\Owner\\OneDrive\\dissertation\\data\\raw\\JSTdatasetR6.xlsx')

print("JST columns:")
print(jst.columns.tolist())
print()
print("JST country identifier sample:")
# Find the country column
country_col = next((c for c in jst.columns if any(kw in c.lower() for kw in ['iso', 'country', 'ctry'])), None)
print(f"Country column found: {country_col}")
if country_col:
    print(jst[country_col].unique())
print()
print("JST year range:")
year_col = next((c for c in jst.columns if 'year' in c.lower()), None)
if year_col:
    print(f"{jst[year_col].min()} - {jst[year_col].max()}")
print()
print("Our ISO codes:")
print(df_merged['iso'].dropna().unique())

JST columns:
['year', 'country', 'iso', 'ifs', 'pop', 'rgdpmad', 'rgdpbarro', 'rconsbarro', 'gdp', 'iy', 'cpi', 'ca', 'imports', 'exports', 'narrowm', 'money', 'stir', 'ltrate', 'hpnom', 'unemp', 'wage', 'debtgdp', 'revenue', 'expenditure', 'xrusd', 'tloans', 'tmort', 'thh', 'tbus', 'bdebt', 'lev', 'ltd', 'noncore', 'crisisJST', 'crisisJST_old', 'peg', 'peg_strict', 'peg_type', 'peg_base', 'JSTtrilemmaIV', 'eq_tr', 'housing_tr', 'bond_tr', 'bill_rate', 'rent_ipolated', 'housing_capgain_ipolated', 'housing_capgain', 'housing_rent_rtn', 'housing_rent_yd', 'eq_capgain', 'eq_dp', 'eq_capgain_interp', 'eq_tr_interp', 'eq_dp_interp', 'bond_rate', 'eq_div_rtn', 'capital_tr', 'risky_tr', 'safe_tr']

JST country identifier sample:
Country column found: country
['Australia' 'Belgium' 'Canada' 'Switzerland' 'Germany' 'Denmark' 'Spain'
 'Finland' 'France' 'UK' 'Ireland' 'Italy' 'Japan' 'Netherlands' 'Norway'
 'Portugal' 'Sweden' 'USA']

JST year range:
1870 - 2020

Our ISO codes:
['US' 'SE' 'AU' '

In [37]:
## # Check JST iso column values
print("JST iso column values:")
print(jst['iso'].unique())
print()

# Create country name to ISO mapping for JST countries
country_to_iso = {
    'Australia': 'AU', 'Belgium': 'BE', 'Canada': 'CA',
    'Switzerland': 'CH', 'Germany': 'DE', 'Denmark': 'DK',
    'Spain': 'ES', 'Finland': 'FI', 'France': 'FR',
    'UK': 'GB', 'Ireland': 'IE', 'Italy': 'IT',
    'Japan': 'JP', 'Netherlands': 'NL', 'Norway': 'NO',
    'Portugal': 'PT', 'Sweden': 'SE', 'USA': 'US'
}

# Add iso to JST if not already matching
jst['iso_match'] = jst['country'].map(country_to_iso)

print("JST countries with ISO mapping:")
print(jst[['country','iso','iso_match']].drop_duplicates().to_string())
print()

# Now merge df_merged with JST using our iso codes
df_check = df_merged[df_merged['iso'].isin(country_to_iso.values())]
print(f"Our speeches matching JST countries: {len(df_check)}")
print(f"Year range in our data: {df_check['year'].min()} - {df_check['year'].max()}")

JST iso column values:
['AUS' 'BEL' 'CAN' 'CHE' 'DEU' 'DNK' 'ESP' 'FIN' 'FRA' 'GBR' 'IRL' 'ITA'
 'JPN' 'NLD' 'NOR' 'PRT' 'SWE' 'USA']

JST countries with ISO mapping:
          country  iso iso_match
0       Australia  AUS        AU
151       Belgium  BEL        BE
302        Canada  CAN        CA
453   Switzerland  CHE        CH
604       Germany  DEU        DE
755       Denmark  DNK        DK
906         Spain  ESP        ES
1057      Finland  FIN        FI
1208       France  FRA        FR
1359           UK  GBR        GB
1510      Ireland  IRL        IE
1661        Italy  ITA        IT
1812        Japan  JPN        JP
1963  Netherlands  NLD        NL
2114       Norway  NOR        NO
2265     Portugal  PRT        PT
2416       Sweden  SWE        SE
2567          USA  USA        US

Our speeches matching JST countries: 7391
Year range in our data: 1997 - 2020


In [39]:
## # Remap our 2-letter ISO to JST 3-letter ISO format
iso_2_to_3 = {
    'AU': 'AUS', 'BE': 'BEL', 'CA': 'CAN', 'CH': 'CHE',
    'DE': 'DEU', 'DK': 'DNK', 'ES': 'ESP', 'FI': 'FIN',
    'FR': 'FRA', 'GB': 'GBR', 'IE': 'IRL', 'IT': 'ITA',
    'JP': 'JPN', 'NL': 'NLD', 'NO': 'NOR', 'PT': 'PRT',
    'SE': 'SWE', 'US': 'USA'
}

df_merged['iso3'] = df_merged['iso'].map(iso_2_to_3)

matched3 = df_merged['iso3'].notna().sum()
print(f"Speeches mapped to JST countries: {matched3:,}")
print(df_merged['iso3'].value_counts())

Speeches mapped to JST countries: 7,391
iso3
USA    2089
DEU     725
JPN     674
GBR     655
CAN     504
AUS     482
CHE     370
FRA     324
ITA     304
ESP     283
NOR     241
IRL     213
FIN     158
NLD     146
DNK      94
PRT      55
BEL      43
SWE      31
Name: count, dtype: int64


In [41]:
## # Override iso column with iso3 to match JST format
df_merged['iso'] = df_merged['iso3']
print("ISO column updated to 3-letter format:")
print(df_merged['iso'].value_counts())

ISO column updated to 3-letter format:
iso
USA    2089
DEU     725
JPN     674
GBR     655
CAN     504
AUS     482
CHE     370
FRA     324
ITA     304
ESP     283
NOR     241
IRL     213
FIN     158
NLD     146
DNK      94
PRT      55
BEL      43
SWE      31
Name: count, dtype: int64


## Cell 5 — Filter to 18 JST countries + 1997–2020, aggregate annually

In [43]:
df_jst = df_merged[
    df_merged['iso'].isin(JST_ISOS) &
    df_merged['year'].between(1997, 2020)
].copy()

print(f'Speeches after JST + date filter : {len(df_jst):,}')
print(f'Countries represented            : {df_jst["iso"].nunique()} / 18')

# Annual mean per country
sentiment_annual = (
    df_jst
    .groupby(['year','iso'])[['P_pos','P_neg','P_neutral']]
    .agg(
        P_pos    =('P_pos',    'mean'),
        P_neg    =('P_neg',    'mean'),
        P_neutral=('P_neutral','mean'),
        n_speeches=('P_neg',  'count')
    )
    .reset_index()
)

# Derived feature
sentiment_annual['net_sentiment'] = (sentiment_annual['P_pos']
                                     - sentiment_annual['P_neg']).round(6)
for col in ['P_pos','P_neg','P_neutral']:
    sentiment_annual[col] = sentiment_annual[col].round(6)

sentiment_annual.to_csv(OUT_ANNUAL, index=False)
print(f'\nAnnual sentiment matrix shape : {sentiment_annual.shape}')
print(f'Expected                      : ~432 rows (18 × 24)')
print(f'Saved → {OUT_ANNUAL}')
print()
print(sentiment_annual.head(12).to_string(index=False))

Speeches after JST + date filter : 7,391
Countries represented            : 18 / 18

Annual sentiment matrix shape : (361, 7)
Expected                      : ~432 rows (18 × 24)
Saved → C:\Users\Owner\OneDrive\dissertation\data\processed\sentiment_annual.csv

 year iso    P_pos    P_neg  P_neutral  n_speeches  net_sentiment
 1997 AUS 0.152554 0.247528   0.599918           7      -0.094974
 1997 CAN 0.317820 0.271316   0.410865           6       0.046504
 1997 CHE 0.131788 0.137297   0.730915           1      -0.005509
 1997 DEU 0.171991 0.076354   0.751656           8       0.095637
 1997 FIN 0.562373 0.109327   0.328300           1       0.453046
 1997 FRA 0.331707 0.410152   0.258141           7      -0.078445
 1997 GBR 0.209627 0.154243   0.636130           8       0.055384
 1997 IRL 0.169480 0.089639   0.740882           1       0.079841
 1997 ITA 0.130515 0.115714   0.753770           3       0.014801
 1997 JPN 0.237929 0.457307   0.304764          21      -0.219378
 1997 NLD 0.19

## Cell 6 — Coverage diagnostic

In [45]:
full_grid = pd.MultiIndex.from_product(
    [range(1997,2021), JST_ISOS], names=['year','iso']
).to_frame(index=False)

check = full_grid.merge(sentiment_annual[['year','iso','P_neg']], on=['year','iso'], how='left')
missing = check[check['P_neg'].isna()]

print(f'Expected country-years : {len(full_grid)}')
print(f'With sentiment data    : {check["P_neg"].notna().sum()}')
print(f'Missing                : {len(missing)}')

if len(missing) > 0:
    print(f'\nMissing country-years:')
    print(missing.pivot_table(index='iso', values='year',
                               aggfunc=['min','max','count']).to_string())
    print('\nGaps filled with per-country mean in Cell 7.')
else:
    print('\n✅  Full 432-row coverage achieved.')

Expected country-years : 432
With sentiment data    : 357
Missing                : 75

Missing country-years:
      min   max count
     year  year  year
iso                  
BEL  1997  2020     7
DNK  1997  2020     6
ESP  1997  2001     5
FIN  1999  2002     3
FRA  1998  2001     3
IRL  1998  2009     8
ITA  2002  2003     2
NLD  1999  2003     4
NOR  1997  2001     4
PRT  1997  2009    12
SWE  2000  2020    21

Gaps filled with per-country mean in Cell 7.


## Cell 7 — Fill gaps and build lagged features (t−1)

In [47]:
# Merge onto the complete 432-row grid
sa = full_grid.merge(sentiment_annual, on=['year','iso'], how='left')

# Fill missing country-years with that country's mean
for col in ['P_pos','P_neg','P_neutral','net_sentiment']:
    country_mean = sa.groupby('iso')[col].transform('mean')
    sa[col] = sa[col].fillna(country_mean)

# Sort for correct lag calculation
sa = sa.sort_values(['iso','year']).reset_index(drop=True)

# Create lagged features — all sentiment used in models must be lagged 1 year
for col in ['P_pos','P_neg','P_neutral','net_sentiment']:
    sa[f'{col}_lag1'] = sa.groupby('iso')[col].shift(1)

lag_cols = [c for c in sa.columns if 'lag1' in c]
print('Lagged features created:', lag_cols)
print(f'NaN in lag cols (1997 rows, expected ~18) : {sa[lag_cols[0]].isna().sum()}')
print()
print(sa[['year','iso','P_neg','P_neg_lag1','net_sentiment_lag1']].head(24).to_string(index=False))

Lagged features created: ['P_pos_lag1', 'P_neg_lag1', 'P_neutral_lag1', 'net_sentiment_lag1']
NaN in lag cols (1997 rows, expected ~18) : 18

 year iso    P_neg  P_neg_lag1  net_sentiment_lag1
 1997 AUS 0.247528         NaN                 NaN
 1998 AUS 0.240418    0.247528           -0.094974
 1999 AUS 0.193348    0.240418           -0.077708
 2000 AUS 0.194881    0.193348           -0.054371
 2001 AUS 0.459429    0.194881           -0.011019
 2002 AUS 0.439675    0.459429           -0.328453
 2003 AUS 0.434833    0.439675           -0.227739
 2004 AUS 0.265144    0.434833           -0.246120
 2005 AUS 0.229180    0.265144           -0.067395
 2006 AUS 0.265760    0.229180           -0.050332
 2007 AUS 0.214351    0.265760           -0.089196
 2008 AUS 0.340125    0.214351            0.028266
 2009 AUS 0.326129    0.340125           -0.156009
 2010 AUS 0.226426    0.326129           -0.088630
 2011 AUS 0.289067    0.226426            0.079182
 2012 AUS 0.217672    0.289067           -

## Cell 8 — Join to JST macro panel → master dataset

In [49]:
# Load JST R6
jst = pd.read_excel(JST_FILE)
jst.columns = jst.columns.str.lower().str.strip()

# Filter to study window
jst_window = jst[jst['year'].between(1997, 2020)].copy()
print(f'JST rows (1997-2020) : {len(jst_window)}')
print(f'JST ISO column name  : {"iso" if "iso" in jst_window.columns else "NOT FOUND — check column names"}')
print(f'JST columns          : {list(jst_window.columns[:10])}')
print()

# Merge on (year, iso)
df_master = jst_window.merge(sa, on=['year','iso'], how='left')

print(f'Master dataset rows  : {len(df_master)}')
print(f'Master dataset cols  : {len(df_master.columns)}')
print(f'P_neg_lag1 coverage  : {df_master["P_neg_lag1"].notna().sum()} / {len(df_master)}')

# Save
df_master.to_csv(OUT_MASTER, index=False)
print(f'\n✅  Master dataset saved → {OUT_MASTER}')

# Final column summary
print(f'\nKey columns available for modelling:')
model_cols = ['year','iso','crisisjst','tloans','ltrate','stir','debtgdp','hpnom',
              'P_neg_lag1','P_pos_lag1','net_sentiment_lag1']
available = [c for c in model_cols if c in df_master.columns]
missing_c = [c for c in model_cols if c not in df_master.columns]
print(f'  Found   : {available}')
if missing_c:
    print(f'  Missing : {missing_c}  ← check JST column names')

JST rows (1997-2020) : 432
JST ISO column name  : iso
JST columns          : ['year', 'country', 'iso', 'ifs', 'pop', 'rgdpmad', 'rgdpbarro', 'rconsbarro', 'gdp', 'iy']

Master dataset rows  : 432
Master dataset cols  : 68
P_neg_lag1 coverage  : 414 / 432

✅  Master dataset saved → C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv

Key columns available for modelling:
  Found   : ['year', 'iso', 'crisisjst', 'tloans', 'ltrate', 'stir', 'debtgdp', 'hpnom', 'P_neg_lag1', 'P_pos_lag1', 'net_sentiment_lag1']


## Cell 9 — Final validation summary

In [51]:
print('=' * 60)
print('  MASTER DATASET VALIDATION SUMMARY')
print('=' * 60)

checks = [
    ('Total rows = 432',       len(df_master) == 432),
    ('Countries = 18',         df_master['iso'].nunique() == 18),
    ('Years = 1997-2020',      df_master['year'].min() == 1997
                               and df_master['year'].max() == 2020),
    ('P_neg_lag1 present',     'P_neg_lag1' in df_master.columns),
    ('crisisJST present',      any('crisis' in c.lower() for c in df_master.columns)),
    ('Macro: tloans present',  any('tloans' in c.lower() for c in df_master.columns)),
]

all_ok = True
for label, result in checks:
    icon = '✅' if result else '❌'
    print(f'  {icon}  {label}')
    if not result:
        all_ok = False

print()
if all_ok:
    print('  ✅  All checks passed.')
    print('  Ready to proceed to model training (Notebook 05).')
else:
    print('  ❌  One or more checks failed — review the cells above.')

print()
print(f'  Output: {OUT_MASTER}')

  MASTER DATASET VALIDATION SUMMARY
  ✅  Total rows = 432
  ✅  Countries = 18
  ✅  Years = 1997-2020
  ✅  P_neg_lag1 present
  ✅  crisisJST present
  ✅  Macro: tloans present

  ✅  All checks passed.
  Ready to proceed to model training (Notebook 05).

  Output: C:\Users\Owner\OneDrive\dissertation\data\processed\jst_sentiment_master.csv
